# 04 — Inventory Optimization
**Project:** Supply Chain Intelligence System  
**Input:** `data/processed/master_orders.csv` + `data/processed/demand_forecast_90day.csv`  
**Goal:** Calculate safety stock, reorder points, and flag SKUs at stockout risk.

---
## Supply Chain Concepts Applied
- **Safety Stock** = Z × σ_demand × √lead_time
- **Reorder Point (ROP)** = (avg_demand × lead_time) + safety_stock
- **Economic Order Quantity (EOQ)** = √(2DS/H)
- **Stockout Risk** = categories where recent demand > historical mean + 1.5σ

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

master   = pd.read_csv('../data/processed/master_orders.csv', parse_dates=['order_purchase_timestamp'])
forecast = pd.read_csv('../data/processed/demand_forecast_90day.csv')
print(f'Master: {master.shape} | Forecast weeks: {len(forecast)}')

## 1. Weekly Demand Statistics by Category

In [ ]:
master['week'] = master['order_purchase_timestamp'].dt.to_period('W').dt.start_time

weekly_by_cat = master.groupby(['week', 'category_en'])['order_id'].nunique().reset_index()
weekly_by_cat.columns = ['week', 'category', 'weekly_orders']

# Demand stats per category
demand_stats = weekly_by_cat.groupby('category')['weekly_orders'].agg(
    avg_demand  = 'mean',
    std_demand  = 'std',
    min_demand  = 'min',
    max_demand  = 'max',
    weeks_active = 'count'
).reset_index()

# Only categories with enough history
demand_stats = demand_stats[demand_stats['weeks_active'] >= 20]

print(f'Categories with sufficient demand history: {len(demand_stats)}')
demand_stats.head()

## 2. Safety Stock Calculation

In [ ]:
# Assumptions (can be adjusted per real business context)
SERVICE_LEVEL   = 0.95   # 95% → Z = 1.645
Z_SCORE         = 1.645
AVG_LEAD_TIME   = 7      # days (seller to carrier avg from EDA)
LEAD_TIME_STD   = 3      # days variability

# Safety Stock = Z × sqrt(lead_time × σ_demand² + avg_demand² × σ_lead²)
demand_stats['safety_stock'] = (
    Z_SCORE * np.sqrt(
        AVG_LEAD_TIME * demand_stats['std_demand']**2 +
        demand_stats['avg_demand']**2 * LEAD_TIME_STD**2
    )
).round(0)

# Reorder Point = (avg_demand_per_day × lead_time) + safety_stock
demand_stats['avg_daily_demand'] = demand_stats['avg_demand'] / 7
demand_stats['reorder_point']    = (
    demand_stats['avg_daily_demand'] * AVG_LEAD_TIME + demand_stats['safety_stock']
).round(0)

# EOQ (simplified — assume holding cost = 20% of unit cost proxy)
# Using revenue proxy: D = annual demand, S = order cost (fixed 50 BRL), H = 20% of avg price
ORDERING_COST   = 50   # BRL per order
HOLDING_PCT     = 0.20
AVG_UNIT_PRICE  = 100  # BRL placeholder

demand_stats['annual_demand'] = demand_stats['avg_demand'] * 52
demand_stats['eoq'] = np.sqrt(
    2 * demand_stats['annual_demand'] * ORDERING_COST /
    (AVG_UNIT_PRICE * HOLDING_PCT)
).round(0)

print('Safety stock + reorder point by top categories:')
print(demand_stats.sort_values('avg_demand', ascending=False).head(10)[
    ['category','avg_demand','std_demand','safety_stock','reorder_point','eoq']
].to_string(index=False))

## 3. Stockout Risk Flags

In [ ]:
# Recent demand = last 4 weeks
recent_cutoff = master['week'].max() - pd.Timedelta(weeks=4)
recent_demand = weekly_by_cat[weekly_by_cat['week'] >= recent_cutoff].groupby('category')['weekly_orders'].mean().reset_index()
recent_demand.columns = ['category', 'recent_avg_demand']

inventory_plan = demand_stats.merge(recent_demand, on='category', how='left')

# Stockout risk: recent demand > historical mean + 1.5σ
inventory_plan['demand_spike'] = (
    inventory_plan['recent_avg_demand'] >
    inventory_plan['avg_demand'] + 1.5 * inventory_plan['std_demand']
)

# Demand trend
inventory_plan['demand_trend'] = (
    (inventory_plan['recent_avg_demand'] - inventory_plan['avg_demand']) /
    inventory_plan['avg_demand'] * 100
).round(1)

# Risk classification
def risk_label(row):
    if row['demand_spike'] and row['demand_trend'] > 30:
        return 'HIGH RISK'
    elif row['demand_trend'] > 15:
        return 'MONITOR'
    elif row['demand_trend'] < -20:
        return 'Slow Moving'
    else:
        return 'Stable'

inventory_plan['stockout_risk'] = inventory_plan.apply(risk_label, axis=1)

print('Stockout risk summary:')
print(inventory_plan['stockout_risk'].value_counts())
print('\nHigh risk categories:')
print(inventory_plan[inventory_plan['stockout_risk']=='HIGH RISK'][['category','avg_demand','recent_avg_demand','demand_trend']].to_string(index=False))

## 4. Export Inventory Plan for Tableau

In [ ]:
inventory_plan.to_csv('../data/processed/inventory_optimization.csv', index=False)
print('Exported: data/processed/inventory_optimization.csv')
print(f'\nFinal shape: {inventory_plan.shape}')
print('\nColumns:', list(inventory_plan.columns))

print('\n=== PROJECT SUMMARY ===')
print(f'Total processed CSVs ready for Tableau:')
import os
for f in os.listdir('../data/processed/'):
    if f.endswith('.csv'):
        path = f'../data/processed/{f}'
        rows = len(pd.read_csv(path))
        print(f'  {f} — {rows} rows')